In [ ]:
##survival:    Survival 
##PassengerId: Unique Id of a passenger. 
##pclass:    Ticket class     
##sex:    Sex     
##Age:    Age in years     
##sibsp:    # of siblings / spouses aboard the Titanic     
##parch:    # of parents / children aboard the Titanic     
##ticket:    Ticket number     
##fare:    Passenger fare     
##cabin:    Cabin number     
##embarked:    Port of Embarkation

In [ ]:
# Importing the libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
dataset= pd.read_csv("../input/train.csv")
test_dataset    = pd.read_csv("../input/test.csv")
titanic = dataset.append(test_dataset, ignore_index=True)
titanic.head()
# create indexes to separate data later on
train_idx = len(dataset)
test_idx = len(titanic) - len(test_dataset)

In [ ]:
# Data Visulization

In [ ]:
fig, axr=plt.subplots(2, 2, figsize=(15, 15))
g=sns.factorplot('Survived', data=dataset, kind="count", hue='Embarked',ax=axr[0][0])
plt.close(g.fig)
g=sns.factorplot('Survived', data=dataset, kind="count", hue='Sex',ax=axr[0][1])
plt.close(g.fig)
g=sns.factorplot('Survived', data=dataset, kind="count", hue='Pclass',ax=axr[1][0])
plt.close(g.fig)
g=sns.factorplot('Survived', data=dataset, kind="count", hue='Parch',ax=axr[1][1])
plt.close(g.fig)
plt.show()

In [ ]:
g = sns.FacetGrid(titanic, col='Survived')
g.map(plt.hist, 'Age', bins=20);

In [ ]:
# Data Preprocessing
# Check null values
titanic.isnull().sum()

In [ ]:
from sklearn.impute import SimpleImputer
my_imputer = SimpleImputer()
data_with_imputed_values = my_imputer.fit_transform(titanic[['Age']])
titanic[['Age']]=data_with_imputed_values


titanic['Embarked'] = titanic['Embarked'].fillna('S')
titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1

In [ ]:
from sklearn.preprocessing import LabelBinarizer
lb = LabelBinarizer()
titanic[['Sex']]=lb.fit_transform(titanic[['Sex']])

In [ ]:
from sklearn.preprocessing import LabelEncoder
lb_make = LabelEncoder()
titanic["Embarked"] = lb_make.fit_transform(titanic["Embarked"])

titanic['Title'] = titanic.Name.apply(lambda name: name.split(',')[1].split('.')[0].strip())
titanic["Title"] = lb_make.fit_transform(titanic["Title"])
titanic.Fare = titanic.Fare.fillna(titanic.Fare.median())

titanic['Deck']=titanic['Cabin'][0]
titanic.Deck = titanic.Deck.fillna('U')

lb = LabelBinarizer()
titanic[['Deck']]=lb.fit_transform(titanic[['Deck']])

titanic=titanic.drop(['Cabin'],axis=1)
titanic=titanic.drop(['Name'],axis=1);
titanic=titanic.drop(['Ticket'],axis=1);

titanic.head()

In [ ]:
# create train and test data
train = titanic[ :train_idx]
test = titanic[test_idx: ]

# convert Survived back to int
train.Survived = train.Survived.astype(int)
# create X and y for data and target values 
X = train.drop(['Survived','PassengerId'], axis=1).values 
y = train.Survived.values


In [ ]:
# create array for test set
X_test = test.drop(['Survived','PassengerId'], axis=1).values

## Model 1: Decision Tree

In [ ]:
def print_score(clf, X_train, y_train, train=True):
    '''
    print the accuracy score, classification report and confusion matrix of classifier
    '''
    if train:
        '''
        training performance
        '''
        print("Train Result:\n")
        y_train_predict=clf.predict(X_train);
        print("accuracy score: {0:.4f}\n".format(accuracy_score(y_train, y_train_predict)))
        print("Classification Report: \n {}\n".format(classification_report(y_train,y_train_predict)))
        print("Confusion Matrix: \n {}\n".format(confusion_matrix(y_train, y_train_predict)))

        res = cross_val_score(clf, X_train, y_train, cv=10, scoring='accuracy')
        print("Average Accuracy: \t {0:.4f}".format(np.mean(res)))
        print("Accuracy SD: \t\t {0:.4f}".format(np.std(res)))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
tree_clf = DecisionTreeClassifier()
tree_clf.fit(X, y);
print_score(tree_clf, X, y, train=True)

## Model 2: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier()
rf_clf.fit(X, y.ravel())
print_score(rf_clf, X, y, train=True)

# Meta Classifier

In [ ]:
en_en = pd.DataFrame()
en_en['tree_clf'] = pd.DataFrame(tree_clf.predict_proba(X))[1]
en_en['rf_clf'] =  pd.DataFrame(rf_clf.predict_proba(X))[1]
col_name = en_en.columns
en_en = pd.concat([en_en, pd.DataFrame(y).reset_index(drop=True)], axis=1)
tmp = list(col_name)
tmp.append('ind')
en_en.columns = tmp
en_en.head()

In [ ]:


from sklearn.linear_model import LogisticRegression
m_clf = LogisticRegression(fit_intercept=False)
m_clf.fit(en_en[['tree_clf', 'rf_clf']], en_en['ind'])
print_score(m_clf, en_en[['tree_clf', 'rf_clf']], en_en['ind'], train=True)

# Single Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier

In [ ]:
pd.Series(list(y)).value_counts() / pd.Series(list(y)).count()
class_weight = {0:0.616162, 1:0.383838}
forest = RandomForestClassifier(class_weight=class_weight)

ada = AdaBoostClassifier(base_estimator=forest, n_estimators=100,
                         learning_rate=0.5, random_state=42)
ada.fit(X, y.ravel())


bag_clf = BaggingClassifier(base_estimator=ada, n_estimators=50,
                            max_samples=1.0, max_features=1.0, bootstrap=True,
                            bootstrap_features=False, n_jobs=-1,
                            random_state=42)
bag_clf.fit(X, y.ravel())
print_score(bag_clf, X, y, train=True)

In [ ]:
# random forrest prediction on test set
rmf_pred = bag_clf.predict(X_test)

In [ ]:
# dataframe with predictions
output = pd.DataFrame({'Survived': rmf_pred})
output['Survived'].value_counts()
test.Survived=rmf_pred
test[['PassengerId','Survived']].to_csv('data_pred.csv', index=False)

In [ ]:
cst=pd.read_csv('data_pred.csv')
cst.head()